# 0. Reestruturação da planilha bruta — Thermal Predictor

O arquivo `assets/others/DataSet_Whrirpool_G03.csv` (exportado da aba **"Option 1 - Last Cases"**, com correções manuais de formatação) está em formato "largo": cada **Run** é uma coluna e cada **termopar (TC)** é uma linha, com as specs de cada run (Reason, Blower Direction, Housing, etc.) empilhadas em linhas de cabeçalho acima da tabela de temperaturas.

Este notebook faz **apenas a reestruturação do formato** (wide → long): nenhum dado é calculado, filtrado ou alterado — só redistribuído. O resultado é um dataframe com uma linha por combinação (Run × TC), contendo as specs do run e do TC como colunas, salvo em `data/dataset_long.csv`.

### Por que reestruturar?

No formato "largo" (wide), cada Run é uma coluna e cada TC é uma linha — ótimo para leitura humana em planilha, mas ruim para análise: para responder algo como "qual foi a temperatura do TC 5 no Run 12?" seria preciso navegar linha/coluna manualmente, e um modelo preditivo não consegue usar a tabela assim, pois espera **uma observação por linha** (aqui, uma combinação Run × TC), com todas as variáveis relevantes (specs do run, specs do TC, temperatura medida) como colunas dessa mesma linha. Essa é a forma "longa" (long format), e é o que este notebook produz.

### Glossário rápido

- **Run**: um experimento/teste específico, com uma combinação de condições (direção do soprador, tipo de housing, etc.).
- **TC (termopar)**: sensor que mede a temperatura em um ponto específico do fogão.
- **Key Area**: região do fogão onde aquele TC está localizado (ex: Mainback, Coil Temps).
- **Active Limit**: temperatura limite considerada segura/aceitável para aquele ponto.
- **Blower Direction / Housing / Board Position**: parâmetros de configuração física do teste (para onde o soprador aponta, se a carcaça está aberta ou fechada, etc.).

## 1. Carregar a planilha bruta

In [1]:
import pandas as pd

raw = pd.read_csv('../assets/others/DataSet_Whrirpool_G03.csv', header=None)
raw.shape

(93, 60)

## 2. Specs por Run

As linhas 0–8 (colunas 4 em diante) trazem, uma spec por linha, os metadados de cada run: rótulo, Reason, Blower Direction, Housing, Board Position, Run Case, Losses, Deflector Hole e Material Coil Support.

Esses índices (linhas 0–8, colunas a partir da 4) não são arbitrários: foram lidos direto da planilha bruta olhando o CSV aberto (`raw.head(10)` mostra as primeiras linhas e confirma onde cada spec está). As colunas 0–3 são reservadas para as specs de TC (seção 3) e só a partir da coluna 4 começam os dados de cada run.

Extraímos esse bloco com `raw.loc[...]` e aplicamos `.T` (transpose) para **girar a tabela**: na planilha bruta cada spec é uma linha e cada run é uma coluna; depois do `.T`, cada run passa a ser uma linha e cada spec uma coluna — formato mais fácil de trabalhar com um dataframe (uma linha = um run).

In [3]:
RUN_SPEC_ROWS = {
    0: 'run_label',
    1: 'reason',
    2: 'blower_direction',
    3: 'housing',
    4: 'board_position',
    5: 'run_case',
    6: 'losses',
    7: 'deflector_hole',
    8: 'material_coil_support',
}
RUN_COL_START = 4

run_block = raw.loc[list(RUN_SPEC_ROWS.keys()), RUN_COL_START:]
run_meta = run_block.T
run_meta.columns = [RUN_SPEC_ROWS[i] for i in run_block.index]
run_meta.index.name = 'col_pos'
run_meta = run_meta.reset_index()
run_meta.head()

,col_pos,run_label,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,4,Run 1,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
1,5,Run 2 - Pyro Flipped,NaN,To Front,Open,Center,Pyro,No Losses,Open,Aluminium
2,6,Run 3 - Pyro Fans in Low,Fan velocity 1500 RPM,To back,Open,Center,Pyro,No Losses,Open,Aluminium
3,7,Run 4 - Broil Case 2,No cover on HS,To back,Open,Center,Broil Case 2,Simone Losses,Open,Aluminium
4,8,Run 5 - Broil Qualitative,No cover on HS,To back,Open,Center,Broil Qualitative,Simone Losses,Open,Aluminium


### 2.1 Desambiguar rótulos de run duplicados

A planilha bruta reusa o mesmo rótulo de run em algumas colunas (ex: "Run 22" aparece 2x). Para gerar um `run_id` único por coluna, mantemos o rótulo original na primeira ocorrência e sufixamos as repetições: `"Run 22"`, `"Run 22 (2)"`.

In [4]:
dup_count = run_meta.groupby('run_label').cumcount()
run_meta['run_id'] = [
    label if dup_count[i] == 0 else f"{label} ({dup_count[i] + 1})"
    for i, label in enumerate(run_meta['run_label'])
]

print(f"{run_meta['run_id'].nunique()} run_id únicos de {len(run_meta)} runs")
run_meta[run_meta['run_label'].duplicated(keep=False)][['col_pos', 'run_label', 'run_id']]

56 run_id únicos de 56 runs


,col_pos,run_label,run_id
5,9,Run 6 - Broil Case 2 - W/ Cover,Run 6 - Broil Case 2 - W/ Cover
7,11,Run 6 - Broil Case 2 - W/ Cover,Run 6 - Broil Case 2 - W/ Cover (2)
21,25,Run 22,Run 22
22,26,Run 22,Run 22 (2)
38,42,Run 43,Run 43
39,43,Run 44,Run 44
40,44,Run 45,Run 45
41,45,Run 44,Run 44 (2)
42,46,Run 43,Run 43 (2)
43,47,Run 44,Run 44 (3)


## 3. Specs por TC (termopar)

A partir da linha 9 (a linha 0–8 é o bloco de specs de run da seção 2 acima), colunas 0–3 trazem `TC Number`, `TC Name`, `Key Area` e `Active Limit` para cada termopar. Intercaladas nessas linhas existem **linhas de cabeçalho de seção** (ex: "Mainback Components", "IPC Components") — são linhas que, na planilha bruta, servem só para separar visualmente os TCs em grupos por região do fogão, e por isso têm quase todas as colunas vazias.

Para detectá-las automaticamente usamos a regra "linha com no máximo 2 valores preenchidos": uma linha de TC normal tem as 4 colunas de spec preenchidas (ou pelo menos 3), enquanto uma linha de cabeçalho de seção só tem o nome da seção na coluna 0 e o resto vazio. Esse limite (`<= 2`) dá uma margem de tolerância para pequenas inconsistências de formatação sem confundir as duas coisas.

Essas linhas de cabeçalho servem apenas para preencher (`ffill`) os poucos `Key Area` vazios dentro de cada bloco — a coluna `Key Area` já vem preenchida na maior parte das linhas.

In [5]:
section_rows = [
    i for i in range(9, raw.shape[0])
    if raw.iloc[i].notna().sum() <= 2 and pd.notna(raw.iloc[i, 0])
]
print("Linhas de cabeçalho de seção:", section_rows)
list(raw.iloc[section_rows, 0])

Linhas de cabeçalho de seção: [9, 21, 30, 38, 44, 82]


['Mainback Components ',
 'Coil Temperatures ',
 'Main Ventilation System',
 'Cooktop/IPC Ventilation System',
 'IPC Components',
 'HMI / Knobs / Console Region ']

In [6]:
tc_rows = [i for i in range(9, raw.shape[0]) if i not in section_rows]

tc_meta = raw.loc[tc_rows, [0, 1, 2, 3]].copy()
tc_meta.columns = ['tc_number', 'tc_name', 'key_area', 'active_limit']

bounds = section_rows + [raw.shape[0]]
for b in range(len(section_rows)):
    start, end = section_rows[b] + 1, bounds[b + 1]
    block_idx = [i for i in tc_meta.index if start <= i < end]
    tc_meta.loc[block_idx, 'key_area'] = tc_meta.loc[block_idx, 'key_area'].ffill()

print("TCs:", tc_meta.shape[0], "| Key Area vazio após ffill:", tc_meta['key_area'].isna().sum())
tc_meta.head()

TCs: 78 | Key Area vazio após ffill: 0


,tc_number,tc_name,key_area,active_limit
10,1,"Door Lock Motor Ambient 1/2"" Below",Mainback,155
11,2,Door Lock Microswitch Body,Mainback,136
12,3,Convect Motor Winding,Mainback,165
13,4,Hot Surface Indicator Body,Mainback,85
14,5,Power On Indicator Body,Mainback,85


## 4. Melt: de largo (Run × TC em matriz) para longo (uma linha por Run × TC)

Esta é a transformação central do notebook. Hoje `temps` é uma matriz: cada linha é um TC, cada coluna é um run, e cada célula é a temperatura medida daquele TC naquele run — o formato "largo" descrito na introdução.

`pd.melt()` "derrete" essa matriz em uma tabela longa: pega cada célula da matriz e transforma em uma linha própria, com uma coluna dizendo de qual run ela veio (`var_name='run_id'`) e outra com o valor da temperatura (`value_name='temp'`). As colunas passadas em `id_vars` (as specs do TC) são repetidas em cada uma dessas novas linhas, porque continuam valendo para aquele TC independente do run.

Depois do melt, cada linha do resultado já identifica o TC (via `id_vars`) e o run (via `run_id`), mas ainda faltam as specs *do run* (Reason, Housing, etc.) — por isso, na sequência, fazemos um `merge` com `run_meta` para trazer essas colunas também, usando `run_id` como chave de junção.

In [7]:
temps = raw.loc[tc_rows, RUN_COL_START:].copy()
temps.columns = run_meta['run_id'].values
temps.index = tc_meta.index

long_df = temps.join(tc_meta).melt(
    id_vars=['tc_number', 'tc_name', 'key_area', 'active_limit'],
    var_name='run_id',
    value_name='temp',
)

run_spec_cols = ['run_id', 'reason', 'blower_direction', 'housing', 'board_position',
                 'run_case', 'losses', 'deflector_hole', 'material_coil_support']
long_df = long_df.merge(run_meta[run_spec_cols], on='run_id', how='left')

# uma célula em branco na planilha original = TC não medido nesse run (não é dado a reestruturar)
long_df = long_df.dropna(subset=['temp']).reset_index(drop=True)

# a planilha bruta usa vírgula como separador decimal (formato pt-BR)
long_df['temp'] = pd.to_numeric(long_df['temp'].str.replace(',', '.', regex=False))

long_df = long_df[[
    'run_id', 'tc_number', 'tc_name', 'key_area', 'active_limit', 'temp',
    'reason', 'blower_direction', 'housing', 'board_position',
    'run_case', 'losses', 'deflector_hole', 'material_coil_support',
]]
long_df.shape

(4065, 14)

## 5. Conferir o resultado

In [8]:
long_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4065 entries, 0 to 4064
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   run_id                 4065 non-null   str    
 1   tc_number              4065 non-null   str    
 2   tc_name                4065 non-null   str    
 3   key_area               4065 non-null   str    
 4   active_limit           4065 non-null   str    
 5   temp                   4065 non-null   float64
 6   reason                 3931 non-null   str    
 7   blower_direction       4065 non-null   str    
 8   housing                4065 non-null   str    
 9   board_position         4065 non-null   str    
 10  run_case               4065 non-null   str    
 11  losses                 3647 non-null   str    
 12  deflector_hole         4065 non-null   str    
 13  material_coil_support  4065 non-null   str    
dtypes: float64(1), str(13)
memory usage: 979.2 KB


In [9]:
long_df.shape

(4065, 14)

In [10]:
long_df.head(20)

,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,Run 1,1,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,99.35,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
1,Run 1,2,Door Lock Microswitch Body,Mainback,136,114.72,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
2,Run 1,3,Convect Motor Winding,Mainback,165,89.93,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
3,Run 1,7,Storage Drawer (Metal Above),Mainback,150,167.87,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
4,Run 1,8,"L2, Wire, Power Cord",Mainback,60,53.65,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
5,Run 1,9,Ambient Below Terminal Block,Mainback,60,48.94,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
6,Run 1,10,ACU Ambient (Hill Only) Top,Mainback,85,60.00,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
7,Run 1,11,ACU K8 Housing / Double Line Break,Mainback,140,55.44,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
8,Run 1,12,Coil FR,Coil Temps,220,55.69,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium
9,Run 1,13,Coil FL,Coil Temps,220,49.33,NaN,To back,Open,Center,Pyro,No Losses,Open,Aluminium


In [11]:
# runs presentes e conferência de que os duplicados foram desambiguados
print(long_df['run_id'].nunique(), "runs distintos")
assert long_df.groupby('run_id').ngroups == run_meta['run_id'].nunique()
sorted(long_df['run_id'].unique())

56 runs distintos


['Run 1',
 'Run 10 - Broil C2 - Closed Housing',
 'Run 11 (DoE Run 2) - Broil Qual - Closed Housing',
 'Run 12- Broil Qual - Top Duct',
 'Run 13 (DoE Run 3)- Broil Qual - Blower to Front Open ',
 'Run 14 (DoE Run 4)- Broil Qual - Blower to Front Closed',
 'Run 16 (DoE Run 6)- Broil Qual - Blower to Front Open',
 'Run 17 (DoE Run 7)- Broil Qual - Blower to Front Open',
 'Run 18 (DoE Run 8)- Broil Qual - Blower to Front Open',
 'Run 19',
 'Run 2 - Pyro Flipped',
 'Run 20',
 'Run 21',
 'Run 22',
 'Run 22 (2)',
 'Run 23',
 'Run 24 - Rerunned',
 'Run 25',
 'Run 26',
 'Run 27',
 'Run 28',
 'Run 29',
 'Run 3 - Pyro Fans in Low',
 'Run 30 ',
 'Run 31 ',
 'Run 32',
 'Run 33',
 'Run 34',
 'Run 35',
 'Run 38',
 'Run 4 - Broil Case 2',
 'Run 41',
 'Run 43',
 'Run 43 (2)',
 'Run 44',
 'Run 44 (2)',
 'Run 44 (3)',
 'Run 45',
 'Run 45 (2)',
 'Run 46',
 'Run 47',
 'Run 48',
 'Run 5 - Broil Qualitative',
 'Run 57',
 'Run 6 - Broil Case 2 - W/ Cover',
 'Run 6 - Broil Case 2 - W/ Cover (2)',
 'Run 60',
 

## 6. Salvar em `data/dataset_long.csv`

In [12]:
from pathlib import Path

caminho_dataset = Path('../data/dataset_long.csv')

long_df.to_csv(caminho_dataset, index=False, encoding='utf-8-sig')
print(f"Salvo em {caminho_dataset.resolve()}: {long_df.shape}")

Salvo em C:\Users\temuj\g03\data\dataset_long.csv: (4065, 14)
